# MedScope AI — Run Entire System on Colab

This notebook runs the entire MedScope AI system (Backend API + Frontend Web App) on Google Colab with a GPU (T4).
It uses `localtunnel` to expose both the API and the web interface to the public internet.

In [ ]:
#@title 1. Runtime Configuration & Mount Drive
import os, sys, shutil
from pathlib import Path

REPO_URL = "https://github.com/prey801/finalyearproject.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
PROJECT_DIR = "/content/finalyearproject"
DRIVE_WEIGHTS_DIR = "/content/drive/MyDrive/medscope-ai/weights" #@param {type:"string"}

# Install localtunnel for exposing ports
!npm install -g localtunnel

# Optional: mount drive for custom SAM2 weights not tracked in git
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    print("Drive not mounted. Will proceed with repo weights only.")

In [ ]:
#@title 2. Clone Repository & Pull Git LFS Weights
!apt-get install -y git-lfs
!git lfs install

if not Path(PROJECT_DIR).exists():
    !git clone --branch $BRANCH $REPO_URL $PROJECT_DIR

%cd $PROJECT_DIR

# Pull large model weights (YOLO & Swin) from Git LFS
print("\nPulling weights from Git LFS...")
!git lfs pull

dst_dir = Path(PROJECT_DIR) / "models/weights"
dst_dir.mkdir(parents=True, exist_ok=True)

# Optionally load custom weights from Google Drive
src_dir = Path(DRIVE_WEIGHTS_DIR)
if src_dir.exists():
    for ext in ["*.pt", "*.pth"]:
        for p in src_dir.glob(ext):
            print(f"Loading {p.name} from Drive...")
            shutil.copy2(p, dst_dir / p.name)

# Check if user trained custom SAM2 weights (starts with sam2 in models/weights)
sam_custom = list(dst_dir.glob("sam2*.pt"))
if not sam_custom:
    sam_weight = Path(PROJECT_DIR) / "sam2_hiera_small.pt"
    if not sam_weight.exists():
        print("\nDownloading default SAM2 segmentation weights...")
        !wget -q -O {str(sam_weight)} https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt
        print("Default SAM2 weights downloaded!")
else:
    print(f"\nFound custom SAM2 weights: {sam_custom[0].name}, skipping default download.")

In [ ]:
#@title 3. Install Backend Dependencies
!apt-get update -qq
!apt-get install -y -qq redis-server postgresql libgl1
!python -m pip install -q -U pip setuptools wheel
!python -m pip install -q -r backend/requirements.txt
!python -m pip install -q -r models/requirements.txt
!python -m pip install -q pytest qdrant-client

In [ ]:
#@title 4. Start Infrastructure (Redis, Postgres, Qdrant)
!service redis-server start
!service postgresql start

# Set up PostgreSQL
!sudo -u postgres psql -c "CREATE USER medscope WITH PASSWORD 'password';" || true
!sudo -u postgres psql -c "CREATE DATABASE medscope_db OWNER medscope;" || true

# Set up Qdrant
if not Path("qdrant").exists():
    !wget -q https://github.com/qdrant/qdrant/releases/download/v1.7.0/qdrant-x86_64-unknown-linux-gnu.tar.gz
    !tar -xzf qdrant-x86_64-unknown-linux-gnu.tar.gz

import subprocess, time
subprocess.Popen(["./qdrant"])
time.sleep(3)
print("Infrastructure services started.")

In [ ]:
#@title 5. Install Frontend Dependencies
%cd $PROJECT_DIR/frontend
!npm install

In [ ]:
#@title 6. Run the Entire System
%cd $PROJECT_DIR
import os, subprocess, time
from pathlib import Path

# Set environment variables for backend
os.environ['YOLO_WEIGHTS_PATH'] = str(Path(PROJECT_DIR) / "models/weights/yolov11_malaria_best.pt")
os.environ['SWIN_WEIGHTS_PATH'] = str(Path(PROJECT_DIR) / "models/weights/swin_malaria_best.pth")

sam_custom = list(Path(PROJECT_DIR).glob("models/weights/sam2*.pt"))
if sam_custom:
    os.environ['SAM2_WEIGHTS_PATH'] = str(sam_custom[0])
else:
    os.environ['SAM2_WEIGHTS_PATH'] = str(Path(PROJECT_DIR) / "sam2_hiera_small.pt")

# 1. Start Backend
print("Starting backend...")
backend_process = subprocess.Popen(["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(5) # Wait for startup

# Expose backend
backend_lt = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE, text=True)
backend_url_line = backend_lt.stdout.readline().strip()
backend_url = backend_url_line.split(" ")[-1] if backend_url_line else "http://localhost:8000"
print(f"Backend exposed at: {backend_url}")

# 2. Configure Frontend with Backend URL
with open(f"{PROJECT_DIR}/frontend/.env.local", "w") as f:
    f.write(f"NEXT_PUBLIC_API_URL={backend_url}\n")

# 3. Start Frontend
print("Starting frontend...")
frontend_process = subprocess.Popen(["npm", "run", "dev"], cwd=f"{PROJECT_DIR}/frontend")
time.sleep(5) # Wait for compilation

# Expose frontend
frontend_lt = subprocess.Popen(["lt", "--port", "3000"], stdout=subprocess.PIPE, text=True)
frontend_url_line = frontend_lt.stdout.readline().strip()
frontend_url = frontend_url_line.split(" ")[-1] if frontend_url_line else "http://localhost:3000"

print("\n" + "="*60)
print(f"🚀 SYSTEM IS RUNNING!")
print(f"🌍 Access the Frontend Web App here: {frontend_url}")
print(f"⚙️ Backend API is at: {backend_url}")
print("="*60 + "\n")

# Keep the cell running so the servers stay alive
try:
    frontend_process.wait()
except KeyboardInterrupt:
    print("\nShutting down...")
    backend_process.terminate()
    frontend_process.terminate()
    backend_lt.terminate()
    frontend_lt.terminate()